# Barrido ε-constraint: **max carbono** s.a. costo ≤ ε
Cada punto maximiza el carbono capturado con un tope de costo ε. ε recorre `[costo del ancla A, costo del ancla B]` en `N_PUNTOS` intervalos iguales, de lo más laxo (ancla B) a lo más estricto (ancla A).

Formulación AUGMECON: `max carbono + ρ·r_carbono·s/r_costo`  s.a.  `costo + s = ε`, `s ≥ 0`.

**Cómo se elige ε (no es arbitrario):** primero se calcula la tabla de pagos con dos anclas lexicográficas
- **A:** min costo → `C_min`; luego max carbono con `costo ≤ C_min`.
- **B:** max carbono → `G_max`; luego min costo con `carbono ≥ G_max`.

La malla de ε divide el rango entre A y B; los extremos de la malla son las propias anclas.
Las anclas se guardan en `resultados_<modo>/anclas/anclas.pkl` y las reutiliza el otro notebook (`barrido_minCosto.ipynb`) si los datos y parámetros no cambiaron.

**AUGMECON2:** el término aumentado evita puntos débilmente eficientes; el *bypass* salta los ε que darían la misma solución.
Cada punto arranca (warm start) desde la mejor solución ya calculada que cumple su ε.

**Salidas** (en `resultados_<modo>/`, con `<modo>` = `selLibre` o `selFija`):
- `frontera_maxCarbono.csv`: un renglón por punto con ε, costo, carbono, holgura, estado, gap, tiempo, dominado y resumen del plan.
- `barrido_maxCarbono/maxCarbono_pXX_calendario.csv`, `_compras.csv`, `_plan_siembra.csv`: plan completo de cada punto.
- `frontera_combinada.csv`: se genera cuando ya existen los dos barridos.

In [14]:
import importlib
import pandas as pd
from IPython.display import display
import modelo_base as mb
importlib.reload(mb)          # para tomar cambios de modelo_base.py sin reiniciar el kernel

<module 'modelo_base' from 'c:\\Users\\REPO\\Tec2026\\Cadena Suministro Reforestacion\\Version4-sumaponderada\\modelo_base.py'>

## Configuración
Todo lo editable vive en `mb.cfg`. El modo (`selFija` o `selLibre`) se cambia **solo en `modelo_base.py`** (`Config.LIBERAR_SELECCION`), para que ambos notebooks usen siempre la misma carpeta de resultados. Después de cambiarlo, reinicia el kernel de los dos notebooks.

In [ ]:
# mb.cfg.CARPETA = r"C:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Modelo3"   # por defecto: carpeta de modelo_base.py
# El modo (selFija / selLibre) se define en modelo_base.py -> Config.LIBERAR_SELECCION
print('Modo de la corrida:', mb.cfg.MODO, '| resultados en', mb.cfg.CARPETA_RESULTADOS)
mb.cfg.N_PUNTOS     = 10      # intervalos de la malla de ε
mb.cfg.TIEMPO_FIJA  = 600     # fase 1 de cada resolución: selección de polígonos fija
mb.cfg.TIEMPO_PUNTO = 900     # fase 2 (solo selLibre): selección libre
mb.cfg.GAP          = 0.01
mb.cfg.RHO          = 1e-3
mb.cfg.USAR_BYPASS  = True


# Decisión a posteriori sobre la frontera
mb.cfg.PESO_CARBONO   = 0.5     # w del carbono (1 - w para el costo)
mb.cfg.PRECIO_CARBONO = None    # $ por unidad de carbono; None = no se usa

Modo de la corrida: selFija | resultados en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija


In [16]:
datos = mb.cargar_datos()
problemas = mb.diagnostico(datos)

Adyacencia: 94 filas en el CSV -> 48 pares únicos
Modo: selFija | resultados en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija
Capacidad: hasta 4.58 ha/día; mínimo exigido 4.0 ha/día
Se necesitan entre 108 y 124 ha (hay 189.5 ha) | compras con llegada entre el día 1 y el 27
Chequeo previo: sin problemas evidentes.


In [17]:
model = mb.construir_modelo(datos)

## Tabla de pagos (anclas A y B)
Si ya existen anclas guardadas con los mismos datos y parámetros, se cargan; si no, se calculan (etapa 0 → costo mínimo → anclas). Para forzar el recálculo usa `recalcular=True`.

In [18]:
anclas = mb.obtener_anclas(model, datos, recalcular=False)

load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.
[etapa0 calendario] maxIterations | costo = n/d | carbono = n/d | gap = 6.45% | 18 s
[costo min (calendario etapa0 fijo)] optimal | costo = 2,424,629.0 | carbono = 798,767.3 | gap = 0.74% | 93 s
load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.
[ancla A paso 1: min costo] maxTimeLimit | costo = 2,424,629.0 | carbono = 798,767.3 | gap = 8.03% | 304 s
load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.
[ancla A paso 2: max carbono | costo <= C_min [fase 1: selección fija]] maxTimeLimit | costo = 2,424,629.0 | carbono = 798,767.3 | gap = 6.03% | 127 s
[ancla B paso 1: max carbono [fase 1: selección fija]] optimal | costo = 2,664,552.5 | carbono = 955,557.4 | gap = 0.00% | 10 s
[ancla B paso 2: min costo | carbono >= G_ma

,ancla,costo,carbono,estado,gap
0,"A (min costo, luego max carbono)",2424629.0,798767.321014,maxTimeLimit,0.06028
1,"B (max carbono, luego min costo)",2646786.5,955557.431871,optimal,0.00827


## Barrido `maxCarbono`

In [19]:
frontera = mb.barrido(model, anclas, "maxCarbono")
if frontera is not None:
    cols = ["punto", "epsilon", "costo", "carbono", "holgura", "estado", "gap", "segundos",
            "dominado", "duplicado", "n_poligonos", "ha_total", "plantado", "edad_promedio", "nota"]
    display(frontera[[c for c in cols if c in frontera.columns]])

Barrido maxCarbono | modo selFija | resultados en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija
17:57:51 inicia maxCarbono p09
load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.
[maxCarbono p09 ε=2,624,570.8 [fase 1: selección fija]] maxTimeLimit | costo = 2,424,629.0 | carbono = 798,767.3 | gap = 16.39% | 128 s
[maxCarbono p08] ε = 2,602,355.0 -> bypass, igual a p09
[maxCarbono p07] ε = 2,580,139.2 -> bypass, igual a p09
[maxCarbono p06] ε = 2,557,923.5 -> bypass, igual a p09
[maxCarbono p05] ε = 2,535,707.8 -> bypass, igual a p09
[maxCarbono p04] ε = 2,513,492.0 -> bypass, igual a p09
[maxCarbono p03] ε = 2,491,276.2 -> bypass, igual a p09
[maxCarbono p02] ε = 2,469,060.5 -> bypass, igual a p09
[maxCarbono p01] ε = 2,446,844.8 -> bypass, igual a p09
Guardado: c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija\frontera

,punto,epsilon,costo,carbono,holgura,estado,gap,segundos,dominado,duplicado,n_poligonos,ha_total,plantado,edad_promedio,nota
0,p00,2424629.00,2424629.0,798767.321014,NaN,maxTimeLimit,0.06028,126.8,False,False,17.0,115.45,54528.0,4.108825,ancla A
1,p01,2446844.75,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
2,p02,2469060.50,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
3,p03,2491276.25,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
4,p04,2513492.00,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
5,p05,2535707.75,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
6,p06,2557923.50,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
7,p07,2580139.25,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
8,p08,2602355.00,2424629.0,798767.321014,NaN,bypass,0.16392,0.0,False,True,NaN,NaN,NaN,NaN,igual a p09 (bypass)
9,p09,2624570.75,2424629.0,798767.321014,199941.75,maxTimeLimit,0.16392,127.8,False,True,17.0,115.45,54528.0,4.108825,


## Frontera combinada
Une ambos barridos, quita dominados y duplicados, y calcula el costo marginal por unidad de carbono y el punto rodilla. Requiere haber corrido también el otro notebook en el mismo modo.

In [20]:
combinada = mb.combinar_fronteras()
if combinada is not None:
    display(combinada[["barrido", "punto", "costo", "carbono", "costo_marginal", "rodilla", "estado", "gap"]])

Combinando fronteras del modo selFija en c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija
No se pueden combinar las fronteras; no existe:
  - c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija\frontera_minCosto.csv
Revisa que ese barrido haya terminado (celda del barrido) y que ambos notebooks usen la misma CARPETA y el mismo LIBERAR_SELECCION.


## Decisión a posteriori: elegir un plan de la frontera
Se evalúa la frontera **ya calculada** (no se vuelve a optimizar):
- **Suma ponderada normalizada:** `max w·z_carbono + (1−w)·z_costo`, con ambos objetivos escalados a [0, 1] usando las anclas.
- **Sensibilidad:** qué punto se elige para cada peso de 0 a 1. Solo los puntos *soportados* (envolvente convexa) pueden ganar.
- **TOPSIS** con los mismos pesos, como contraste.
- **Precio del carbono** (opcional): `max p·carbono − costo`, un peso con sentido económico.

In [21]:
resultado = mb.decision_a_posteriori()
if resultado is not None:
    decision, sensibilidad = resultado
    display(decision[["barrido", "punto", "costo", "carbono", "soportado", "score_ponderado", "topsis", "elegido_por"]])

No existe c:\Users\REPO\Tec2026\Cadena Suministro Reforestacion\Version4-sumaponderada\resultados_selFija\frontera_combinada.csv: corre primero ambos barridos y combinar_fronteras().
